In [2]:
import pandas as pd
import joblib

In [6]:
recommend_model = joblib.load("C:\\Users\\ayush\\Desktop\\College\\5th sem\\Advance ML\\Smart Crop\\Models\\crop_recommendation_model.pkl")

yield_model = joblib.load("C:\\Users\\ayush\\Desktop\\College\\5th sem\\Advance ML\\Smart Crop\\Models\\yield_prediction_model.pkl")

In [7]:
label_encoder = joblib.load("../models/label_encoder.pkl")

state_encoder = joblib.load("../models/state_encoder.pkl")

district_encoder = joblib.load("../models/district_encoder.pkl")

season_encoder = joblib.load("../models/season_encoder.pkl")

crop_encoder = joblib.load("../models/crop_encoder.pkl")

In [8]:
url = "https://raw.githubusercontent.com/shreyzo/Crop-yield-and-profitability-prediction/main/datafile.csv"

df_profit = pd.read_csv(url)

In [9]:
df_profit.head()

,Crop,State,Cost of Cultivation (`/Hectare) A2+FL,Cost of Cultivation (`/Hectare) C2,Cost of Production (`/Quintal) C2,Yield (Quintal/ Hectare),Support price
0,ARHAR,Uttar Pradesh,9794.05,23076.74,1941.55,9.83,6000
1,ARHAR,Karnataka,10593.15,16528.68,2172.46,7.47,6000
2,ARHAR,Gujarat,13468.82,19551.90,1898.30,9.59,6000
3,ARHAR,Andhra Pradesh,17051.66,24171.65,3670.54,6.42,6000
4,ARHAR,Maharashtra,17130.55,25270.26,2775.80,8.72,6000


In [10]:
df_profit = df_profit.rename(columns={
    "Cost of Cultivation (`/Hectare) A2+FL": "Cost_Cultivation_A2FL",
    "Cost of Cultivation (`/Hectare) C2": "Cost_Cultivation_C2",
    "Cost of Production (`/Quintal) C2": "Cost_Production_C2",
    "Yield (Quintal/ Hectare) ": "Yield_Quintal_Hectare",
    "Support price": "Support_Price"
})

In [11]:
# Revenue per Hectare

df_profit["Revenue_per_Hectare"] = (
    df_profit["Support_Price"] *
    df_profit["Yield_Quintal_Hectare"]
)

In [12]:
# Profit per Hectare

df_profit["Profit_per_Hectare"] = (
    df_profit["Revenue_per_Hectare"] -
    df_profit["Cost_Cultivation_C2"]
)

In [13]:
def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    

    input_data = pd.DataFrame({
        "N": [N],
        "P": [P],
        "K": [K],
        "temperature": [temperature],
        "humidity": [humidity],
        "ph": [ph],
        "rainfall": [rainfall]
    })

    
    prediction = recommend_model.predict(input_data)

    
    crop = label_encoder.inverse_transform(prediction)

    return crop[0]

In [14]:
def predict_yield(
    state,
    district,
    year,
    season,
    crop,
    temperature,
    humidity,
    soil_moisture
):

    try:
        state = state_encoder.transform([state])[0]
        district = district_encoder.transform([district])[0]
        season = season_encoder.transform([season])[0]
        crop = crop_encoder.transform([crop])[0]

    except ValueError:
        return None

    input_data = pd.DataFrame({
        "State_Name_encoded": [state],
        "District_Name_encoded": [district],
        "Season_encoded": [season],
        "Crop_encoded": [crop],
        "Crop_Year": [year],
        "Temperature": [temperature],
        "Humidity": [humidity],
        "Soil_Moisture": [soil_moisture]
    })

    prediction = yield_model.predict(input_data)

    return round(float(prediction[0]), 2)

In [15]:
def get_profit_estimate(crop, state):
    # Convert crop name to uppercase
    crop = crop.upper()

    # Filter matching row
    result = df_profit[
        (df_profit["Crop"] == crop) &
        (df_profit["State"] == state)
    ]

    # If no matching row
    if result.empty:
        return None

    # Get first matching row
    row = result.iloc[0]

    # Return required information
    return {
        "Crop": row["Crop"],
        "State": row["State"],
        "Support Price": row["Support_Price"],
        "Cost of Cultivation (C2)": row["Cost_Cultivation_C2"],
        "Historical Yield": row["Yield_Quintal_Hectare"],
        "Profit per Hectare": row["Profit_per_Hectare"]
    }

In [16]:
profit_crop_mapping = {

    "rice": "PADDY",

    "maize": "MAIZE",

    "cotton": "COTTON",

    "mungbean": "MOONG",

    "pigeonpeas": "ARHAR",

    "mothbeans": None,

    "kidneybeans": None,

    "blackgram": None,

    "coffee": None,

    "banana": None,

    "apple": None,

    "grapes": None,

    "papaya": None,

    "orange": None,

    "watermelon": None,

    "muskmelon": None,

    "mango": None,

    "jute": None,

    "coconut": None
}

## Smart Advisory Function

In [17]:
def smart_crop_advisory(
    N,
    P,
    K,
    temperature,
    humidity,
    ph,
    rainfall,
    state,
    district,
    year,
    season,
    soil_moisture
):

    print("=" * 60)
    print("SMART CROP ADVISORY SYSTEM")
    print("=" * 60)

    # ==========================
    # Step A : Recommend Crop
    # ==========================

    recommended_crop = recommend_crop(
        N,
        P,
        K,
        temperature,
        humidity,
        ph,
        rainfall
    )

    print(f"Recommended Crop : {recommended_crop}")

    # ==========================
    # Step B : Convert for Module 2
    # ==========================

    crop_for_yield = recommended_crop.strip().title()

    # ==========================
    # Step C : Predict Yield
    # ==========================

    predicted_yield = predict_yield(
        state=state,
        district=district,
        year=year,
        season=season,
        crop=crop_for_yield,
        temperature=temperature,
        humidity=humidity,
        soil_moisture=soil_moisture
    )

    if predicted_yield is None:
        print("Predicted Yield : Not Available")
    else:
        print(f"Predicted Yield : {predicted_yield:.2f}")

    # ==========================
    # Step D : Profit Estimation
    # ==========================

    profit_crop = profit_crop_mapping.get(recommended_crop)

    if profit_crop is None:

        profit = None

    else:

        profit = get_profit_estimate(
            profit_crop.lower(),
            state
        )

    # ==========================
    # Step E : Print Profit
    # ==========================

    if profit is None:

        print("Profit Information : Not Available")

    else:

        print(f"Support Price : ₹{profit['Support Price']}")
        print(f"Historical Yield : {profit['Historical Yield']}")
        print(f"Estimated Profit per Hectare : ₹{profit['Profit per Hectare']:.2f}")

    print("=" * 60)

## Test the adviser

In [18]:
smart_crop_advisory(
    N=90,
    P=42,
    K=43,
    temperature=21,
    humidity=82,
    ph=6.5,
    rainfall=203,

    state="Andhra Pradesh",
    district="ANANTAPUR",

    year=2023,
    season="Kharif",

    soil_moisture=45
)

SMART CROP ADVISORY SYSTEM
Recommended Crop : rice
Predicted Yield : 3.07
Support Price : ₹1868
Historical Yield : 56.0
Estimated Profit per Hectare : ₹58157.80


In [20]:
smart_crop_advisory(
    N=70,
    P=50,
    K=45,
    temperature=24,
    humidity=65,
    ph=6.8,
    rainfall=85,

    state="Andhra Pradesh",
    district="GUNTUR",

    year=2023,
    season="Kharif",

    soil_moisture=38
)

SMART CROP ADVISORY SYSTEM
Recommended Crop : maize
Predicted Yield : 4.42
Support Price : ₹1850
Historical Yield : 42.68
Estimated Profit per Hectare : ₹41156.15


In [21]:
smart_crop_advisory(
    N=101,
    P=17,
    K=47,
    temperature=24,
    humidity=70,
    ph=6.0,
    rainfall=160,

    state="Andhra Pradesh",
    district="ANANTAPUR",

    year=2023,
    season="Kharif",

    soil_moisture=45
)


SMART CROP ADVISORY SYSTEM
Recommended Crop : coffee
Predicted Yield : Not Available
Profit Information : Not Available
